In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
class TwoLayerNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TwoLayerNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.sigmoid(x)
        x = self.fc2(x)
        return x

def train_model(X_train, y_train, X_val, y_val, input_size, hidden_size=16, output_size=1, n_epochs=100000, batch_size=100, learning_rate=0.02, patience=20):
    model = TwoLayerNet(input_size, hidden_size, output_size)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            with torch.no_grad():
                train_outputs = model(X_train)
                train_loss = criterion(train_outputs, y_train).item()
                print(f"Epoch {epoch}, train loss: {train_loss}, val loss: {val_loss}")

    model.load_state_dict(best_model_state)
    return model


start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

predictions = []
dates = []
y_test_list = []

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train_val = train_data.drop(['permno', 'DATE', 'RET'], axis=1)
    y_train_val = train_data['RET']
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    X_train_val = torch.tensor(X_train_val.values, dtype=torch.float32)
    y_train_val = torch.tensor(y_train_val.values.reshape(-1, 1), dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)
    
 
    train_size = int(0.8 * len(X_train_val))
    X_train = X_train_val[:train_size]
    y_train = y_train_val[:train_size]
    X_val = X_train_val[train_size:]
    y_val = y_train_val[train_size:]
    input_size = X_train.shape[1]
    model = train_model(X_train, y_train, X_val, y_val, input_size)
    model.eval()
    with torch.no_grad():
        preds = model(X_test).numpy()
    
    predictions.append(preds)
    dates.append(test_data['DATE'].values)
    y_test_list.append(y_test.numpy())
predictions_all = np.concatenate(predictions, axis=0)
y_test_list_all = np.concatenate(y_test_list, axis=0)
dates_all = np.concatenate(dates, axis=0)
R2OOS = 1 - sum((y_test_list_all - predictions_all)**2) / sum(y_test_list_all**2)
print("R2OOS: ", R2OOS)